# STG_8.2 — Serializar artefactos con features de autoría (spec 014)

Esta notebook es la versión "con autoría" de `STG_8` (spec 010): reutiliza el modelo
ganador ya validado en la spec 013 (`STG_6.2`/`STG_7.2`) — LightGBM por defecto, Escenario A
(PRO fuera de la coalición de Milei) — y deja en disco todo lo que la API necesita para
predecir con el autor del proyecto como dato de entrada.

No entrena ni evalúa nada nuevo: la métrica oficial para citar sigue siendo **F1-macro =
0.581** (holdout genuino de `STG_6.2`). Todos los artefactos se guardan con **nombres
nuevos** (sufijo `_autor`) — no se pisa ningún archivo de las specs 009/010/013.

Tareas de `specs/014-modelo-autor-en-app/tasks.md` cubiertas acá: T1 a T8.

## T1 — Nómina canónica de 257 diputados

Hoy la predicción muestra 259 personas en vez de 257 porque dos diputados aparecen con dos
grafías distintas de su nombre (una vieja, con datos de bloque desactualizados, y una
actual): **Osuna, Blanca Inés** y **Pichetto, Miguel Angel**. Esta celda construye la nómina
"canónica": una fila por cada uno de los 257 diputados de `diputados_actuales.csv` (la
nómina oficial), con el nombre exactamente como aparece en su voto **más reciente** del
dataset, y la lista de todas las grafías históricas que hay que sumar para reconstruir su
historial completo.

El cruce nómina oficial ↔ dataset de votos usa la misma normalización que el resto del
proyecto (sin tildes, mayúsculas, espacios colapsados), extendida para **quitar comillas**:
`diputados_actuales.csv` tiene el campo de "Alí, Ernesto \"Pipi\"" mal escapado como CSV (las
comillas internas del apodo no llevan el escape `""` que exige el formato), así que sin
quitar comillas de ambos lados ese único diputado no matchea contra el dataset. La quita de
comillas se usa **solo para la clave de cruce**, no para el nombre que se muestra.

In [1]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import unicodedata
import json

import pandas as pd
import numpy as np
import joblib

DATA_DIR = Path("../data")
RANDOM_STATE = 42


def normalizar_nombre(s):
    """Mayusculas, sin tildes, sin caracteres de encoding roto, espacios colapsados
    (misma funcion que STG_5.2 -- se repite aca para no crear una dependencia entre
    notebooks)."""
    if pd.isna(s):
        return ''
    s = str(s).upper().strip()
    s = s.replace('�', 'N')
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return ' '.join(s.split())


def clave_match(s):
    """Extiende normalizar_nombre quitando comillas, solo para EMPAREJAR nombres entre
    diputados_actuales.csv (nomina oficial) y el dataset de votos -- ver celda anterior
    (caso Ali, Ernesto "Pipi"). No se usa para el nombre canonico que se muestra."""
    return normalizar_nombre(s).replace('"', '')


# --- Nomina oficial: fuente del universo de 257 y del bloque/provincia vigente ---
oficial = pd.read_csv(DATA_DIR / "diputados_actuales.csv")
oficial['nombre_completo'] = oficial['Apellido'].str.strip() + ', ' + oficial['Nombre'].str.strip()
oficial['clave'] = oficial['nombre_completo'].map(clave_match)

assert len(oficial) == 257, f"Se esperaban 257 diputados en la nomina oficial, hay {len(oficial)}"
assert oficial['clave'].is_unique, "Hay nombres oficiales duplicados tras normalizar"

# --- Grafias del dataset de votos: se usa df_consolidado.csv (el historico COMPLETO, la
# misma fuente que hoy lee api/database.py para el historial de un diputado) para no dejar
# afuera ninguna grafia que solo aparezca en votaciones filtradas fuera de df_modelado ---
df_consolidado = pd.read_csv(DATA_DIR / "df_consolidado.csv", parse_dates=["fecha_votacion"])
grafias = df_consolidado[['diputado', 'fecha_votacion']].copy()
grafias['clave'] = grafias['diputado'].map(clave_match)

# Nombre canonico de cada clave = grafia de su voto MAS RECIENTE en el dataset completo
canonico_por_clave = (
    grafias.sort_values('fecha_votacion')
    .groupby('clave')
    .tail(1)
    .set_index('clave')['diputado']
    .to_dict()
)

oficial['diputado_canonico'] = oficial['clave'].map(canonico_por_clave)
sin_match = oficial[oficial['diputado_canonico'].isna()]
print("Diputados de la nomina oficial SIN voto registrado en el dataset:")
print(sin_match[['Apellido', 'Nombre']].to_string() if len(sin_match) else "(ninguno)")
assert len(sin_match) == 0, "Hay diputados de la nomina oficial sin match -- revisar normalizacion"

# --- Grafias historicas que absorbe cada canonico (para combinar el historial completo) ---
claves_oficiales = set(oficial['clave'])
grafias_por_clave = (
    grafias[grafias['clave'].isin(claves_oficiales)]
    .groupby('clave')['diputado']
    .agg(lambda s: sorted(set(s)))
)

nomina = oficial[['clave', 'diputado_canonico', 'Bloque', 'Distrito']].rename(
    columns={'Bloque': 'bloque_actual', 'Distrito': 'provincia_actual'}
)
nomina['grafias_historicas'] = nomina['clave'].map(grafias_por_clave)
nomina['n_grafias'] = nomina['grafias_historicas'].map(len)

assert len(nomina) == 257
assert nomina['diputado_canonico'].is_unique, "Dos diputados oficiales distintos canonizaron al mismo nombre"

print()
print(f"Nomina canonica construida: {len(nomina)} diputados")
print(f"Diputados con mas de una grafia en el historial: {(nomina['n_grafias'] > 1).sum()}")
print(nomina.loc[nomina['n_grafias'] > 1, ['diputado_canonico', 'bloque_actual', 'grafias_historicas']].to_string())
print()
print("T1 PASA: 257/257 diputados de la nomina oficial matchean con el dataset de votos.")

Diputados de la nomina oficial SIN voto registrado en el dataset:
(ninguno)

Nomina canonica construida: 257 diputados
Diputados con mas de una grafia en el historial: 2
          diputado_canonico        bloque_actual                                grafias_historicas
170      OSUNA, BLANCA INES  UNIÓN POR LA PATRIA          [OSUNA, BLANCA INES, OSUNA, Blanca Inés]
188  PICHETTO, MIGUEL ANGEL    ENCUENTRO FEDERAL  [PICHETTO, MIGUEL ANGEL, PICHETTO, Miguel Angel]

T1 PASA: 257/257 diputados de la nomina oficial matchean con el dataset de votos.


In [2]:
# Exportar la nomina canonica a disco -- la usan la API (T9, para /diputados y el historial
# unificado) y el resto de esta notebook (T2 en adelante)
nomina_export = nomina.copy()
nomina_export['grafias_historicas'] = nomina_export['grafias_historicas'].map(lambda gs: ';'.join(gs))
nomina_export = nomina_export.rename(columns={'diputado_canonico': 'diputado'})[
    ['diputado', 'clave', 'bloque_actual', 'provincia_actual', 'grafias_historicas', 'n_grafias']
]
nomina_export.to_csv(DATA_DIR / "nomina_diputados.csv", index=False)

assert len(nomina_export) == 257
print(f"Guardado: {DATA_DIR / 'nomina_diputados.csv'} ({len(nomina_export)} diputados)")

Guardado: ..\data\nomina_diputados.csv (257 diputados)


## T2 — Snapshot de features del diputado (historial combinado, sin `.shift`)

Mismo criterio que `STG_8` T5 (spec 010): las tasas históricas de cada diputado se acumulan
con **todo** su historial conocido hasta hoy, sin excluir ningún voto — no es leakage, porque
al predecir una ley hipotética futura no hay ningún "voto actual" que haya que descartar.

La diferencia con `STG_8` es que acá, **antes de agrupar**, cada fila se reetiqueta con el
nombre **canónico** del diputado (usando la clave normalizada de T1). Así el historial de
Osuna y Pichetto —hoy repartido en dos grafías— se combina en una sola fila por persona. La
fuente es `df_modelado_autor.csv` (28.738 filas, mismas que `df_modelado.csv` + columnas de
autoría de las specs 011/012), filtrando AUSENTE igual que siempre (no es una posición
política).

El bloque/provincia "actual" de cada diputado sigue siendo el de su voto más reciente — ya
verificado en T1 que coincide con el bloque oficial de `diputados_actuales.csv` para los 257.
La codificación usa el **mismo encoder** ya ajustado en `STG_5.3` (`encoder_bloque_provincia_autor.joblib`), sin reajustar.

In [3]:
CORTE_2023 = pd.Timestamp('2023-12-10')
CORTE_2026 = pd.Timestamp('2026-01-01')

df_votos_raw = pd.read_csv(DATA_DIR / "df_modelado_autor.csv", parse_dates=["fecha_votacion"])

# Conteo de votos ANTES de canonizar, para el assert de no-perdida de historial (Osuna/Pichetto)
n_votos_antes = df_votos_raw['diputado'].value_counts()

df_votos_raw = df_votos_raw[df_votos_raw['voto'] != 'AUSENTE'].copy()
df_votos_raw['voto'] = df_votos_raw['voto'].replace('ABSTENCION', 'ABSTENCIÓN')
assert len(df_votos_raw) == 25082, "El filtro de AUSENTE no dio el mismo resultado que STG_5/STG_8"

# Reetiquetar con el nombre CANONICO antes de agrupar -- esto es lo que combina el historial
# de Osuna y Pichetto en una sola fila cada uno (T1 ya verifico 257/257 de match)
df_votos_raw['diputado_clave'] = df_votos_raw['diputado'].map(clave_match)
assert df_votos_raw['diputado_clave'].isin(claves_oficiales).all(), \
    "Hay votos de diputados que no estan en la nomina oficial de 257 -- revisar filtro previo"
df_votos_raw['diputado_canonico'] = df_votos_raw['diputado_clave'].map(
    nomina.set_index('clave')['diputado_canonico']
)

df_votos_raw['es_afirmativo'] = (df_votos_raw['voto'] == 'AFIRMATIVO').astype(float)

# Voto mayoritario del BLOQUE (bloque tal cual en cada fila, al momento de ese voto -- no
# depende de la canonizacion del nombre del diputado)
voto_bloque = (
    df_votos_raw.groupby(['titulo_base', 'fecha_votacion', 'bloque'])['voto']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={'voto': 'voto_bloque'})
)
df_votos_raw = df_votos_raw.merge(voto_bloque, on=['titulo_base', 'fecha_votacion', 'bloque'], how='left')
df_votos_raw['alineado_bloque'] = (df_votos_raw['voto'] == df_votos_raw['voto_bloque']).astype(float)

# Acumulado TOTAL por diputado CANONICO (sin shift: no hay voto "actual" que excluir)
snapshot_diputado = df_votos_raw.groupby('diputado_canonico').agg(
    tasa_afirmativo_hist=('es_afirmativo', 'mean'),
    tasa_alineacion_bloque_hist=('alineado_bloque', 'mean'),
    n_votos_hist=('es_afirmativo', 'size'),
)

for col_out, corte in [('tasa_afirmativo_desde_2023', CORTE_2023), ('tasa_afirmativo_2026', CORTE_2026)]:
    ventana = df_votos_raw[df_votos_raw['fecha_votacion'] >= corte]
    snapshot_diputado[col_out] = ventana.groupby('diputado_canonico')['es_afirmativo'].mean()

# Bloque/provincia actuales = los del voto mas reciente de cada diputado canonico
ultimo_voto = (
    df_votos_raw.sort_values('fecha_votacion')
    .groupby('diputado_canonico')
    .tail(1)
    .set_index('diputado_canonico')
)
snapshot_diputado['bloque'] = ultimo_voto['bloque']
snapshot_diputado['provincia'] = ultimo_voto['provincia']

# Encoding con el MISMO encoder ya ajustado en STG_5.3 (no se reajusta)
enc_votante = joblib.load(DATA_DIR / "encoder_bloque_provincia_autor.joblib")
snapshot_diputado[['bloque_enc', 'provincia_enc']] = enc_votante.transform(
    snapshot_diputado[['bloque', 'provincia']]
)

snapshot_diputado = snapshot_diputado.reset_index().rename(columns={'diputado_canonico': 'diputado'})
snapshot_diputado.to_csv(DATA_DIR / "snapshot_diputado_autor.csv", index=False)

print(f"Diputados en el snapshot: {len(snapshot_diputado)}")
assert len(snapshot_diputado) == 257, f"Se esperaban 257 diputados en el snapshot, hay {len(snapshot_diputado)}"

# Verificacion: el historial combinado de Osuna/Pichetto = suma de lo que estaba repartido
# entre sus dos grafias (no se perdio ni se duplico nada al canonizar)
for clave_dup in ['OSUNA, BLANCA INES', 'PICHETTO, MIGUEL ANGEL']:
    grafias_dup = nomina.loc[nomina['clave'] == clave_dup, 'grafias_historicas'].iloc[0]
    esperado_total_raw = sum(n_votos_antes.get(g, 0) for g in grafias_dup)
    # esperado_total_raw incluye AUSENTE; recalculamos el esperado post-filtro de AUSENTE
    esperado_sin_ausente = df_votos_raw[df_votos_raw['diputado'].isin(grafias_dup)].shape[0]
    obtenido = int(snapshot_diputado.loc[snapshot_diputado['diputado'] == nomina.loc[nomina['clave']==clave_dup, 'diputado_canonico'].iloc[0], 'n_votos_hist'].iloc[0])
    print(f"{clave_dup}: grafias={grafias_dup} | votos sin AUSENTE esperados={esperado_sin_ausente} | n_votos_hist obtenido={obtenido}")
    assert obtenido == esperado_sin_ausente, f"Se perdio o duplico historial al canonizar {clave_dup}"

print()
print("NaN por columna (esperado: puede haber NaN en las tasas por ventana si el diputado")
print("no tiene votos en ese periodo; se resuelven con la cascada de relleno en la API):")
print(snapshot_diputado.isna().sum().to_string())
print()
print(f"Guardado: {DATA_DIR / 'snapshot_diputado_autor.csv'}")
print()
print("T2 PASA: 257 diputados, historial de Osuna/Pichetto combinado y verificado.")

Diputados en el snapshot: 257
OSUNA, BLANCA INES: grafias=['OSUNA, BLANCA INES', 'OSUNA, Blanca Inés'] | votos sin AUSENTE esperados=382 | n_votos_hist obtenido=382
PICHETTO, MIGUEL ANGEL: grafias=['PICHETTO, MIGUEL ANGEL', 'PICHETTO, Miguel Angel'] | votos sin AUSENTE esperados=139 | n_votos_hist obtenido=139

NaN por columna (esperado: puede haber NaN en las tasas por ventana si el diputado
no tiene votos en ese periodo; se resuelven con la cascada de relleno en la API):
diputado                       0
tasa_afirmativo_hist           0
tasa_alineacion_bloque_hist    0
n_votos_hist                   0
tasa_afirmativo_desde_2023     0
tasa_afirmativo_2026           1
bloque                         0
provincia                      0
bloque_enc                     0
provincia_enc                  0

Guardado: ..\data\snapshot_diputado_autor.csv

T2 PASA: 257 diputados, historial de Osuna/Pichetto combinado y verificado.


## T3 — Verificar que el bloque del snapshot coincide con el oficial

T2 ya asignó a cada diputado el bloque/provincia de su voto más reciente y los codificó. Esta
celda es el chequeo cruzado: confirma que ese bloque "del último voto" coincide (normalizado)
con el `Bloque` oficial de `diputados_actuales.csv` para los 257. Si algún día deja de
cumplirse —por ejemplo, si el dataset de votos queda desactualizado respecto a un cambio de
bloque reciente— el `assert` lo va a atajar acá, listando los casos puntuales para decidir a
mano en vez de dejar pasar una inconsistencia en silencio.

In [4]:
comparacion_bloque = nomina[['diputado_canonico', 'bloque_actual']].merge(
    snapshot_diputado[['diputado', 'bloque']],
    left_on='diputado_canonico', right_on='diputado', how='left'
)
comparacion_bloque['bloque_oficial_norm'] = comparacion_bloque['bloque_actual'].map(normalizar_nombre)
comparacion_bloque['bloque_snapshot_norm'] = comparacion_bloque['bloque'].map(normalizar_nombre)

divergentes = comparacion_bloque[
    comparacion_bloque['bloque_oficial_norm'] != comparacion_bloque['bloque_snapshot_norm']
]

print(f"Diputados verificados: {len(comparacion_bloque)}")
print(f"Divergencias entre bloque oficial y bloque del ultimo voto: {len(divergentes)}")
if len(divergentes):
    print(divergentes[['diputado_canonico', 'bloque_actual', 'bloque']].to_string())

assert len(divergentes) == 0, (
    "Hay diputados cuyo bloque oficial (diputados_actuales.csv) no coincide con el bloque "
    "de su voto mas reciente en el dataset -- revisar los casos listados arriba antes de "
    "seguir, puede ser un cambio de bloque no reflejado todavia en el historico de votos."
)

print()
print("T3 PASA: el bloque del snapshot (ultimo voto) coincide con el oficial para los 257.")

Diputados verificados: 257
Divergencias entre bloque oficial y bloque del ultimo voto: 0

T3 PASA: el bloque del snapshot (ultimo voto) coincide con el oficial para los 257.


## T4 — Snapshots por tema (diputado × tema y bloque × tema)

Mismo criterio que `STG_8` T6/T7: para cada combinación (diputado, tema) y (bloque, tema) se
calcula la tasa de AFIRMATIVO usando todo el historial conocido, sin `.shift`. Se reutiliza
`df_votos_raw` tal cual quedó después de T2 (ya filtrado, con `diputado_canonico`) y se le
suma el `tema_id` de cada título (mismo merge que hace `STG_5`/`STG_8`). El snapshot por
bloque usa la columna `bloque` tal cual está en cada fila (no depende de la canonización del
nombre del diputado).

In [5]:
df_temas_map = pd.read_csv(DATA_DIR / "df_features_titulo.csv")[["titulo_base", "tema_id"]]

df_votos_tema = df_votos_raw.merge(df_temas_map, on="titulo_base", how="left")
assert df_votos_tema["tema_id"].isna().sum() == 0, "Quedaron titulos sin tema_id tras el merge"

# --- snapshot_diputado_tema_autor: tasa de AFIRMATIVO por (diputado_canonico, tema) ---
snapshot_diputado_tema = (
    df_votos_tema.groupby(["diputado_canonico", "tema_id"])["es_afirmativo"]
    .mean()
    .reset_index()
    .rename(columns={"es_afirmativo": "tasa_afirmativo_tema_hist", "diputado_canonico": "diputado"})
)
snapshot_diputado_tema.to_csv(DATA_DIR / "snapshot_diputado_tema_autor.csv", index=False)

combinaciones_posibles = snapshot_diputado["diputado"].nunique() * 20
print(f"Combinaciones diputado-tema con historial: {len(snapshot_diputado_tema):,} de {combinaciones_posibles:,} posibles")
print(f"NaN en tasa_afirmativo_tema_hist: {snapshot_diputado_tema['tasa_afirmativo_tema_hist'].isna().sum()}")
assert snapshot_diputado_tema['diputado'].nunique() <= 257
print(f"Guardado: {DATA_DIR / 'snapshot_diputado_tema_autor.csv'}")

# --- snapshot_bloque_tema_autor: tasa de AFIRMATIVO por (bloque, tema) ---
snapshot_bloque_tema = (
    df_votos_tema.groupby(["bloque", "tema_id"])["es_afirmativo"]
    .mean()
    .reset_index()
    .rename(columns={"es_afirmativo": "tasa_afirmativo_bloque_tema"})
)
snapshot_bloque_tema.to_csv(DATA_DIR / "snapshot_bloque_tema_autor.csv", index=False)

combinaciones_posibles_bloque = df_votos_tema["bloque"].nunique() * 20
print()
print(f"Combinaciones bloque-tema con historial: {len(snapshot_bloque_tema):,} de {combinaciones_posibles_bloque:,} posibles")
print(f"NaN en tasa_afirmativo_bloque_tema: {snapshot_bloque_tema['tasa_afirmativo_bloque_tema'].isna().sum()}")
print(f"Guardado: {DATA_DIR / 'snapshot_bloque_tema_autor.csv'}")

assert snapshot_diputado_tema['tasa_afirmativo_tema_hist'].isna().sum() == 0
assert snapshot_bloque_tema['tasa_afirmativo_bloque_tema'].isna().sum() == 0
print()
print("T4 PASA.")

Combinaciones diputado-tema con historial: 2,521 de 5,140 posibles
NaN en tasa_afirmativo_tema_hist: 0
Guardado: ..\data\snapshot_diputado_tema_autor.csv

Combinaciones bloque-tema con historial: 571 de 1,100 posibles
NaN en tasa_afirmativo_bloque_tema: 0
Guardado: ..\data\snapshot_bloque_tema_autor.csv

T4 PASA.


## T5 — Orden exacto de las 398 features del Escenario A

**Punto crítico**: en `STG_6.2`, las features del Escenario A no siguen el orden de las
columnas del CSV — se arman como `FEATURES_COMUNES + ["es_oficialista",
"coincide_bloque_autor"]`, es decir, **las dos features del escenario van al final**, no en
la posición que ocupan dentro de `df_entrenamiento_autor.csv`. Si la API reconstruyera el
orden simplemente leyendo el encabezado del CSV, desalinearía esas dos columnas contra lo
que el modelo aprendió — y predeciría mal **en silencio**, sin ningún error visible. Por eso
acá se reproduce textualmente la misma expresión de `STG_6.2` y el resultado se guarda como
artefacto (`orden_features_autor.json`), para que la API nunca tenga que adivinar el orden.

In [6]:
# Reproduccion textual de la construccion de FEATURES_A en STG_6.2 (celda T11 de esa notebook)
columnas_entrenamiento = pd.read_csv(DATA_DIR / "df_entrenamiento_autor.csv", nrows=0).columns

META   = ["diputado", "titulo_base", "fecha_votacion", "id_votacion", "bloque", "provincia", "tema_label"]
TARGET = "voto"

FEATURES_COMUNES = [c for c in columnas_entrenamiento if c not in META + [TARGET]
                     and c not in ("es_oficialista", "coincide_bloque_autor",
                                    "es_oficialista_b", "coincide_bloque_autor_b")]

FEATURES_A = FEATURES_COMUNES + ["es_oficialista", "coincide_bloque_autor"]

print(f"Columnas totales en df_entrenamiento_autor.csv: {len(columnas_entrenamiento)}")
print(f"Features comunes  : {len(FEATURES_COMUNES)}")
print(f"Features Escenario A: {len(FEATURES_A)} (ultimas 2: {FEATURES_A[-2:]})")

assert len(FEATURES_A) == 398, f"Se esperaban 398 features del Escenario A, hay {len(FEATURES_A)}"

# Verificacion una a una: FEATURES_A debe ser exactamente el mismo conjunto de columnas que
# usa el modelo (todas las columnas del CSV salvo META/TARGET/las 2 del escenario B)
esperado = set(columnas_entrenamiento) - set(META) - {TARGET, "es_oficialista_b", "coincide_bloque_autor_b"}
assert set(FEATURES_A) == esperado, "FEATURES_A no coincide exactamente con las columnas de entrenamiento del Escenario A"
assert len(FEATURES_A) == len(set(FEATURES_A)), "Hay columnas repetidas en FEATURES_A"

with open(DATA_DIR / "orden_features_autor.json", "w", encoding="utf-8") as f:
    json.dump(FEATURES_A, f, ensure_ascii=False, indent=2)

print()
print(f"Guardado: {DATA_DIR / 'orden_features_autor.json'}")
print()
print("T5 PASA: 398 features del Escenario A, orden identico a STG_6.2, verificado una a una.")

Columnas totales en df_entrenamiento_autor.csv: 408
Features comunes  : 396
Features Escenario A: 398 (ultimas 2: ['es_oficialista', 'coincide_bloque_autor'])

Guardado: ..\data\orden_features_autor.json

T5 PASA: 398 features del Escenario A, orden identico a STG_6.2, verificado una a una.


## T6 — Reentrenar el LGBM ganador (Escenario A) sobre todo el dataset

Mismos hiperparámetros que ganaron en `STG_6.2` (sin volver a tunear — `STG_7.2` ya confirmó
que el afinado no mejora sobre el default): `LGBMClassifier(class_weight='balanced',
random_state=42, n_jobs=-1, verbosity=-1)`. Se entrena con **todas** las filas de
`df_entrenamiento_autor.csv` (20.608, sin partición train/holdout) para que el modelo que va
a servir la API aproveche toda la información disponible — igual que hizo `STG_8` con el
modelo de la spec 009. Esto no es una evaluación nueva ni leakage: no se mide nada acá, el
holdout genuino ya cumplió su función en `STG_6.2` (F1-macro = 0.581).

Se reusa el `LabelEncoder` ya ajustado en `STG_5.3` (`le_voto_autor.joblib`), sin reajustar,
para que las clases codificadas sean las mismas que aprendió el modelo original.

In [7]:
from lightgbm import LGBMClassifier

df_train = pd.read_csv(DATA_DIR / "df_entrenamiento_autor.csv", parse_dates=["fecha_votacion"])
assert len(df_train) == 20608, f"Se esperaban 20.608 filas, hay {len(df_train)}"

le_voto = joblib.load(DATA_DIR / "le_voto_autor.joblib")
df_train["voto_enc"] = le_voto.transform(df_train[TARGET])

X_full = df_train[FEATURES_A].values
y_full = df_train["voto_enc"].values

modelo_lgbm_autor = LGBMClassifier(
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)
modelo_lgbm_autor.fit(X_full, y_full)

joblib.dump(modelo_lgbm_autor, DATA_DIR / "modelo_lgbm_autor.joblib")

print(f"Modelo entrenado sobre {len(df_train):,} filas (Escenario A, todas las filas disponibles).")
print(f"Clases: {list(le_voto.classes_)}")
print(f"Features usadas: {len(FEATURES_A)}")
print(f"Guardado en: {DATA_DIR / 'modelo_lgbm_autor.joblib'}")
print()
print("T6 PASA.")

Modelo entrenado sobre 20,608 filas (Escenario A, todas las filas disponibles).
Clases: ['ABSTENCIÓN', 'AFIRMATIVO', 'NEGATIVO']
Features usadas: 398
Guardado en: ..\data\modelo_lgbm_autor.joblib

T6 PASA.


## T7 — Chequeo de sanidad contra el holdout de `STG_6.2`

**Importante para la defensa del TP**: este número **no es una métrica de generalización
nueva** y no reemplaza al **0.581** documentado en la spec 013. El modelo de T6 se entrenó
con **todas** las filas (incluido el holdout de `STG_6.2`), así que evaluarlo sobre esas
mismas filas es evaluarlo sobre datos que ya vio — no es una prueba honesta de "qué tan bien
predice datos nuevos". Ese resultado honesto ya se midió correctamente en `STG_6.2` con un
modelo que **no** había visto el holdout, y es el que hay que citar siempre:
**F1-macro holdout = 0.581**.

Lo único que este chequeo verifica es que **serializar no rompió nada**: mismo orden de
columnas, mismo encoder de clases, mismo comportamiento del modelo. Por eso se espera que el
resultado sea igual o mejor al 0.581 (el modelo ahora conoce esas filas); si diera **peor**,
sería señal de un bug real (columnas desalineadas, encoder distinto, etc.) y habría que
frenar antes de seguir. Se reconstruye el mismo split 80/20 por fecha que usó `STG_6.2`.

In [8]:
from sklearn.metrics import f1_score

# Mismo split 80/20 por fecha que STG_6.2 (recordatorio: aca es solo para el sanity check,
# el modelo de T6 ya fue entrenado incluyendo estas filas)
df_sorted = df_train.sort_values("fecha_votacion").reset_index(drop=True)
corte_idx = int(len(df_sorted) * 0.80)
fecha_corte = df_sorted.iloc[corte_idx]["fecha_votacion"]
df_hold = df_sorted.iloc[corte_idx:].copy()

X_hold = df_hold[FEATURES_A].values
y_hold = df_hold["voto_enc"].values

y_pred_hold = modelo_lgbm_autor.predict(X_hold)
f1_hold_final = f1_score(y_hold, y_pred_hold, average='macro')

F1_HOLDOUT_STG62 = 0.581  # documentado en specs/013-reentrenar-modelo-features-autor

print(f"Fecha de corte            : {fecha_corte.date()}")
print(f"Filas en el holdout       : {len(df_hold):,}")
print(f"F1-macro (STG_6.2, honesto, modelo SIN ver el holdout): {F1_HOLDOUT_STG62}")
print(f"F1-macro (este chequeo, modelo CON el holdout ya visto): {f1_hold_final:.3f}")
print()

assert f1_hold_final >= F1_HOLDOUT_STG62, (
    f"El modelo final da PEOR F1-macro ({f1_hold_final:.3f}) que el modelo original "
    f"({F1_HOLDOUT_STG62}) a pesar de haber visto estos datos. Esto indica un bug en la "
    f"serializacion (columnas desalineadas, encoder distinto, hiperparametros mal copiados, etc.)."
)

print("T7 PASA: el refit final no rompio nada. El F1-macro oficial para citar en la defensa")
print(f"sigue siendo el de STG_6.2 (holdout genuino, sin ver esos datos): {F1_HOLDOUT_STG62}.")

Fecha de corte            : 2025-12-18
Filas en el holdout       : 4,122
F1-macro (STG_6.2, honesto, modelo SIN ver el holdout): 0.581
F1-macro (este chequeo, modelo CON el holdout ya visto): 0.928

T7 PASA: el refit final no rompio nada. El F1-macro oficial para citar en la defensa
sigue siendo el de STG_6.2 (holdout genuino, sin ver esos datos): 0.581.


## T8 — Resumen: por qué esta notebook cumple la Constitución

Cada paso ya trae su propia explicación en la celda correspondiente. Este resumen las junta
para que el equipo pueda defenderlo de un vistazo, principio por principio.

**Principio 2 (validación temporal) y Principio 4 (F1-macro)** — Esta notebook **no evalúa un
modelo nuevo**. El único número de tipo F1-macro que aparece (T7, 0.928) es un chequeo interno
de que la serialización no rompió nada — se documenta explícitamente que **no reemplaza** al
0.581 de `STG_6.2`, que sigue siendo la métrica oficial del proyecto (medida con un modelo
que nunca vio ese holdout).

**Principio 3 (cero data leakage)** — Los puntos donde podría colarse fuga de información y
por qué no ocurre:
- *Refit del LGBM sobre todas las filas (T6)*: no hay ninguna evaluación nueva — el holdout
  ya cumplió su función en `STG_6.2`/`STG_7.2`. Reentrenar con todo el historial para
  producción es práctica estándar, no una trampa metodológica.
- *Snapshots (T2, T4)*: se calculan como acumulado de **toda** la historia conocida, sin
  `.shift(1)` — correcto porque no hay ningún "voto actual" que excluir al predecir una ley
  hipotética futura (mismo criterio ya aceptado en `STG_8`, spec 010).
- *Nómina canónica (T1)*: el nombre canónico de cada diputado es el de su voto más reciente
  **dentro del historial ya conocido**, no depende de ningún dato futuro.

**Principio 5 (reproducibilidad)** — `random_state=42` fijo en el LGBM (T6). Los artefactos
se guardan con nombre propio, sin sobrescribir ningún dato de las specs 009/010/013:

| Artefacto | Contenido |
|---|---|
| `data/modelo_lgbm_autor.joblib` | LGBM ganador (Escenario A), reentrenado sobre todas las filas |
| `data/orden_features_autor.json` | Las 398 features del Escenario A, en el orden exacto que espera el modelo |
| `data/snapshot_diputado_autor.csv` | Tasas históricas por diputado (257, historial de grafías combinado) |
| `data/snapshot_diputado_tema_autor.csv` | Tasa de AFIRMATIVO por (diputado, tema) |
| `data/snapshot_bloque_tema_autor.csv` | Tasa de AFIRMATIVO por (bloque, tema) |

Se reutilizan sin cambios (no listados arriba): `kmeans_temas.joblib`, `mapa_temas.json`,
`le_voto_autor.joblib`, `encoder_bloque_autor.joblib`, `encoder_bloque_provincia_autor.joblib`
— ya serializados en `STG_8`/`STG_5.3`.

**Principio 7 (simple antes que sofisticado)** — No se agregó ninguna librería nueva ni ningún
modelo nuevo: se reutiliza exactamente lo ya validado en la spec 013.

**Reproducibilidad verificada**: la notebook se corrió completa dos veces de punta a punta y
los 6 artefactos nuevos salieron **idénticos byte a byte** (verificado con hash de archivo).